# Same-season calibration and development uncertainty

Calibration is evaluated only on earlier sessions from the same season. Offering-clustered bootstrap intervals show whether small point-estimate improvements are distinguishable from sampling noise.

In [1]:
from __future__ import annotations
import hashlib, inspect, json, os, platform, sys
from pathlib import Path
import joblib, numpy as np, pandas as pd, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name == 'v2' or not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data' / 'Enrollment-Data-master'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'v2'
CACHE_ROOT = ARTIFACT_ROOT / 'cache'
MODEL_ROOT = PROJECT_ROOT / 'model'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 20260812
SESSION_ORDER = ['20229','20235','20239','20245','20249','20255','20259','20265']
SEASONS = {'fall_winter':['20229','20239','20249','20259'], 'summer':['20235','20245','20255','20265']}
FINAL_TEST = {'fall_winter':'20259', 'summer':'20265'}
DEVELOPMENT = {k:[s for s in v if s != FINAL_TEST[k]] for k,v in SEASONS.items()}

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def fingerprint(payload) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def versions():
    return {'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
            'scikit_learn':sklearn.__version__,'joblib':joblib.__version__}
BASE = ['position_to_capacity','waitlist_to_capacity','days_to_deadline','movement_3d','movement_7d','position','waitlist','capacity','capacity_changed_7d','position_to_waitlist','days_squared','log_waitlist','movement_velocity_7d']
CONTEXT = BASE + ['near_deadline_7d','days_under_7','days_under_14','days_over_60','position_ratio_near_7d','waitlist_ratio_near_7d','rank_over_30pct','campus_erin','campus_scar','term_winter','term_full_year','winter_near_7d','scar_near_7d']
RANK_MONOTONIC={'position_to_capacity':-1,'position':-1,'position_to_waitlist':-1,'position_ratio_near_7d':-1,'rank_over_30pct':-1}
def targeted(frame):
    f=frame.copy(); days=f.days_to_deadline.astype(float); near=(days<=7).astype('float32')
    f['near_deadline_7d']=near; f['days_under_7']=(7-days).clip(lower=0); f['days_under_14']=(14-days).clip(lower=0); f['days_over_60']=(days-60).clip(lower=0)
    f['position_ratio_near_7d']=f.position_to_capacity*near; f['waitlist_ratio_near_7d']=f.waitlist_to_capacity*near
    f['rank_over_30pct']=(f.position_to_capacity>.30).astype('float32'); f['campus_erin']=(f.campus=='ERIN').astype('float32'); f['campus_scar']=(f.campus=='SCAR').astype('float32')
    f['term_winter']=(f.term=='winter').astype('float32'); f['term_full_year']=(f.term=='full_year').astype('float32')
    f['winter_near_7d']=f.term_winter*near; f['scar_near_7d']=f.campus_scar*near
    return f
def make_model(features, *, leaf=15, l2=3.0):
    constraints=[RANK_MONOTONIC.get(x,0) for x in features]
    return Pipeline([('features',ColumnTransformer([('numeric',SimpleImputer(strategy='median'),features)],remainder='drop')),
      ('model',HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,max_leaf_nodes=leaf,l2_regularization=l2,monotonic_cst=constraints,random_state=RANDOM_STATE))])
def ece(y,p,w,bins=10):
    edges=np.linspace(0,1,bins+1); ids=np.clip(np.digitize(p,edges)-1,0,bins-1); total=w.sum(); out=0
    for b in range(bins):
        m=ids==b
        if m.any(): out+=w[m].sum()/total*abs(np.average(y[m],weights=w[m])-np.average(p[m],weights=w[m]))
    return float(out)
def metrics(frame,p):
    y=frame.cleared.to_numpy(); w=frame.model_weight.to_numpy(); p=np.clip(np.asarray(p),1e-6,1-1e-6)
    auc=roc_auc_score(y,p,sample_weight=w) if np.unique(y).size>1 else np.nan
    return {'brier':brier_score_loss(y,p,sample_weight=w),'log_loss':log_loss(y,p,sample_weight=w,labels=[0,1]),'ece':ece(y,p,w),'auc':auc,'accuracy':np.average((p>=.5)==y,weights=w)}

from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression


## Load the uniquely fingerprinted development decision

In [2]:
cache_manifest=json.loads((ARTIFACT_ROOT/'cache-manifest.json').read_text()); current_cache_fingerprint=fingerprint(cache_manifest)
candidates=[]
for candidate_path in ARTIFACT_ROOT.glob('development-selection-*.json'):
    candidate=json.loads(candidate_path.read_text())
    if candidate.get('cache_fingerprint')==current_cache_fingerprint and 'paired_brier_differences_vs_base' in candidate: candidates.append((candidate_path,candidate))
if not candidates: raise RuntimeError('No development selection matches the current cache and paired-selection contract')
selection_path,selection=max(candidates,key=lambda item:item[0].stat().st_mtime_ns); spec=selection['spec']; features=spec['features']
CACHE_VERSION=int(cache_manifest['cache_version'])
session_samples={s:targeted(pd.read_pickle(CACHE_ROOT/f'{s}-positions-v{CACHE_VERSION}.pkl')) for sessions in DEVELOPMENT.values() for s in sessions}
assert selection['cache_fingerprint']==current_cache_fingerprint


## Same-season calibration experiment

In [3]:
def logit(p):
    p=np.clip(p,1e-6,1-1e-6)
    return np.log(p/(1-p)).reshape(-1,1)

def fit_calibrator(method, probability, frame):
    if method == 'none': return {'method':'none','fit':'temporal_same_season_oof'}
    if method.startswith('platt_C'):
        C=float(method.split('C',1)[1])
        model=LogisticRegression(C=C,solver='lbfgs').fit(logit(probability),frame.cleared,sample_weight=frame.model_weight)
        return {'method':'platt','coefficient':float(model.coef_[0,0]),'intercept':float(model.intercept_[0]),'C':C,'fit':'temporal_same_season_oof'}
    if method == 'isotonic_same_season':
        model=IsotonicRegression(out_of_bounds='clip').fit(probability,frame.cleared,sample_weight=frame.model_weight)
        return {'method':'isotonic','x':model.X_thresholds_.astype(float).tolist(),'y':model.y_thresholds_.astype(float).tolist(),'fit':'temporal_same_season_oof'}
    raise ValueError(method)

def apply_calibrator(parameters, probability):
    p=np.clip(np.asarray(probability),1e-6,1-1e-6)
    if parameters['method']=='none': return p
    if parameters['method']=='platt':
        score=parameters['coefficient']*logit(p).ravel()+parameters['intercept']
        return 1/(1+np.exp(-score))
    if parameters['method']=='isotonic':
        return np.interp(p,parameters['x'],parameters['y'],left=parameters['y'][0],right=parameters['y'][-1])
    raise ValueError(parameters['method'])

def slice_gap(frame,p,mask):
    mask=np.asarray(mask)
    if not mask.any() or mask.all(): return np.nan
    return float(abs(np.average(frame.cleared[mask],weights=frame.model_weight[mask])-np.average(np.asarray(p)[mask],weights=frame.model_weight[mask])))

def maximum_probability_bin_gap(frame,p,min_weight_fraction=.02):
    bins=pd.cut(p,np.linspace(0,1,11),include_lowest=True); total=frame.model_weight.sum(); gaps=[]
    for _,group in frame.assign(_p=p,_bin=bins).groupby('_bin',observed=True):
        if group.model_weight.sum()/total < min_weight_fraction: continue
        gaps.append(abs(np.average(group.cleared,weights=group.model_weight)-np.average(group._p,weights=group.model_weight)))
    return float(max(gaps)) if gaps else np.inf

CALIBRATION_METHODS=['none','platt_C0.03','platt_C0.1','platt_C0.3','platt_C1.0','isotonic_same_season']
rows=[]; predictions=[]; calibration_parameters={}
for season,sessions in DEVELOPMENT.items():
    # Generate temporal out-of-fold predictions from expanding same-season fits.
    folds=[]
    for fold in range(1,len(sessions)):
        train=pd.concat([session_samples[s] for s in sessions[:fold]],ignore_index=True); valid=session_samples[sessions[fold]]
        model=make_model(**spec); model.fit(train[features],train.cleared,model__sample_weight=train.model_weight)
        folds.append({'session':sessions[fold],'frame':valid,'probability':model.predict_proba(valid[features])[:,1]})
    calibration_fold,validation_fold=folds[0],folds[-1]
    calibration_frame=calibration_fold['frame']; val=validation_fold['frame']; p_raw=validation_fold['probability']
    if calibration_frame.cleared.nunique()<2: raise RuntimeError(f'{season}: calibration fold has one outcome class')
    candidate_probabilities={}
    for method in CALIBRATION_METHODS:
        provisional=fit_calibrator(method,calibration_fold['probability'],calibration_frame)
        p=apply_calibrator(provisional,p_raw); candidate_probabilities[method]=p
        rows.append({'season':season,'method':method,**metrics(val,p),
          'near_deadline_gap':slice_gap(val,p,val.days_to_deadline.le(7)),
          'large_queue_gap':slice_gap(val,p,val.waitlist.ge(100)),
          'max_probability_gap':maximum_probability_bin_gap(val,p)})
    predictions.append({'season':season,'frame':val,'probabilities':candidate_probabilities})
    all_oof_probability=np.concatenate([fold['probability'] for fold in folds]); all_oof=pd.concat([fold['frame'] for fold in folds],ignore_index=True)
    if all_oof.cleared.nunique()<2: raise RuntimeError(f'{season}: cannot fit calibration with one outcome class')
    calibration_parameters[season]={method:fit_calibrator(method,all_oof_probability,all_oof) for method in CALIBRATION_METHODS}
calibration_scores=pd.DataFrame(rows); calibration_scores


,season,method,brier,log_loss,ece,auc,accuracy,near_deadline_gap,large_queue_gap,max_probability_gap
0,fall_winter,none,0.119347,0.380071,0.015365,0.766413,0.843651,0.040760,0.004221,0.022866
1,fall_winter,platt_C0.03,0.119174,0.379966,0.011018,0.766413,0.843687,0.020254,0.011329,0.015682
2,fall_winter,platt_C0.1,0.119118,0.379602,0.009964,0.766413,0.843936,0.016701,0.009651,0.022522
3,fall_winter,platt_C0.3,0.119107,0.379513,0.009782,0.766413,0.844119,0.015607,0.009156,0.023317
4,fall_winter,platt_C1.0,0.119103,0.379483,0.009633,0.766413,0.844200,0.015216,0.008981,0.023553
5,fall_winter,isotonic_same_season,0.119593,0.381996,0.015298,0.765447,0.843137,0.025348,0.000822,0.027992
6,summer,none,0.130545,0.408414,0.027055,0.835200,0.814529,0.029968,0.034425,0.059068
7,summer,platt_C0.03,0.130870,0.412525,0.034446,0.835200,0.815285,0.044858,0.000207,0.120172
8,summer,platt_C0.1,0.129651,0.406978,0.018186,0.835200,0.815993,0.078496,0.008324,0.068204
9,summer,platt_C0.3,0.129661,0.406231,0.018110,0.835200,0.815284,0.090245,0.011531,0.049745


## Offering-clustered bootstrap intervals

In [4]:
def offering_errors(frame,p):
    work=pd.DataFrame({'offering_id':frame.offering_id.to_numpy(),'weight':frame.model_weight.to_numpy(),'error':frame.model_weight.to_numpy()*(frame.cleared.to_numpy()-np.asarray(p))**2})
    return work.groupby('offering_id',sort=False).agg(error=('error','sum'),weight=('weight','sum')).to_numpy().T
def bootstrap_from_offerings(error,weight,reps=1000):
    rng=np.random.default_rng(RANDOM_STATE); chosen=rng.integers(0,len(error),size=(reps,len(error)))
    values=error[chosen].sum(axis=1)/weight[chosen].sum(axis=1)
    return np.quantile(values,[.025,.5,.975]).tolist()
def clustered_bootstrap(frame,p,reps=1000): return bootstrap_from_offerings(*offering_errors(frame,p),reps)
def paired_clustered_bootstrap(frame,left,right,reps=1000):
    left_error,weight=offering_errors(frame,left); right_error,_=offering_errors(frame,right)
    return bootstrap_from_offerings(left_error-right_error,weight,reps)
uncertainty=[]
for item in predictions:
    season=item['season']; val=item['frame']; probabilities=item['probabilities']; raw=probabilities['none']
    for method,p in probabilities.items():
        uncertainty.append({'season':season,'method':method,'brier_ci':clustered_bootstrap(val,p)})
        if method!='none': uncertainty.append({'season':season,'comparison':f'{method}_minus_none','method':method,'paired_brier_difference_ci':paired_clustered_bootstrap(val,p,raw)})
uncertainty


[{'season': 'fall_winter',
  'method': 'none',
  'brier_ci': [0.11497214286271795, 0.11910243433185405, 0.12381581916765888]},
 {'season': 'fall_winter',
  'method': 'platt_C0.03',
  'brier_ci': [0.11511916284913795, 0.1189694555497038, 0.12326325973185598]},
 {'season': 'fall_winter',
  'comparison': 'platt_C0.03_minus_none',
  'method': 'platt_C0.03',
  'paired_brier_difference_ci': [-0.0005537977938746172,
   -0.00016550016839462925,
   0.00019253764652281969]},
 {'season': 'fall_winter',
  'method': 'platt_C0.1',
  'brier_ci': [0.11505478630829814, 0.11892930549058313, 0.12319100338745109]},
 {'season': 'fall_winter',
  'comparison': 'platt_C0.1_minus_none',
  'method': 'platt_C0.1',
  'paired_brier_difference_ci': [-0.0006436745847804424,
   -0.00021867364350186755,
   0.0001789944017346476]},
 {'season': 'fall_winter',
  'method': 'platt_C0.3',
  'brier_ci': [0.11504439487835447, 0.11892204493498088, 0.12317387763964223]},
 {'season': 'fall_winter',
  'comparison': 'platt_C0.3_mi

## Lock calibration and release thresholds before latest-session evaluation

In [ ]:
RELEASE_THRESHOLDS={'oracle_brier_lt_literal':True,'ece_lte':.05,'near_deadline_gap_lte':.08,'large_queue_gap_lte':.08,'max_probability_gap_lte':.10,'probability_bin_min_weight_fraction':.02,'minimum_class_weight_fraction':.01}
BRIER_NONINFERIORITY_MARGIN=.002
paired_ci={(item['season'],item['method']):item['paired_brier_difference_ci'] for item in uncertainty if 'paired_brier_difference_ci' in item}
calibration={}; calibration_selection=[]
for season in DEVELOPMENT:
    season_scores=calibration_scores.loc[calibration_scores.season.eq(season)].copy()
    eligible=[]
    for row in season_scores.itertuples():
        brier_ok=row.method=='none' or paired_ci[(season,row.method)][2] <= BRIER_NONINFERIORITY_MARGIN
        diagnostics_ok=(row.ece<=RELEASE_THRESHOLDS['ece_lte'] and row.near_deadline_gap<=RELEASE_THRESHOLDS['near_deadline_gap_lte'] and
          row.large_queue_gap<=RELEASE_THRESHOLDS['large_queue_gap_lte'] and row.max_probability_gap<=RELEASE_THRESHOLDS['max_probability_gap_lte'])
        calibration_selection.append({'season':season,'method':row.method,'brier_noninferior':bool(brier_ok),'diagnostics_pass':bool(diagnostics_ok),'eligible':bool(brier_ok and diagnostics_ok)})
        if brier_ok and diagnostics_ok: eligible.append(row)
    if not eligible: raise RuntimeError(f'{season}: no calibration candidate passes the pre-final development requirements')
    # Calibration is chosen by ECE among Brier-noninferior candidates; Brier breaks ties.
    selected=min(eligible,key=lambda row:(row.ece,row.brier,row.method))
    calibration[season]=selected.method
selected_parameters={season:calibration_parameters[season][method] for season,method in calibration.items()}
locked={'selection_fingerprint':selection['fingerprint'],'cache_fingerprint':current_cache_fingerprint,'spec':spec,'calibration':calibration,
 'calibration_parameters':selected_parameters,'calibration_selection':calibration_selection,'brier_noninferiority_margin':BRIER_NONINFERIORITY_MARGIN,
 'scores':calibration_scores.to_dict('records'),'uncertainty':uncertainty,'release_thresholds':RELEASE_THRESHOLDS,'versions':versions()}
locked['fingerprint']=fingerprint(locked)
for old_path in ARTIFACT_ROOT.glob('locked-spec-*.json'): old_path.unlink()
path=ARTIFACT_ROOT/f'locked-spec-{locked["fingerprint"]}.json'; path.write_text(json.dumps(locked,indent=2),encoding='utf-8')
locked, path


## Interpretation

Calibration is selected entirely on earlier same-season sessions. A candidate must be Brier-noninferior to the raw model under an offering-clustered paired interval and must pass the same calibration, near-deadline, large-queue, and probability-bin diagnostics used at release. Among eligible candidates, the lowest development ECE wins. Release thresholds and the chosen mapping are locked before the latest completed Fall/Winter and Summer sessions are loaded.
